## LangChain 파이프라인 심화

앞의 `02_Langchain_basic.ipynb`에서는 `create_agent()`로 **LLM이 스스로 도구를 골라 쓰는 에이전트**를 만들었습니다.

그런데 업무에는 순서가 이미 정해져 있는 일도 많습니다. 예를 들어 인수인계 메모를 받으면

**메모 읽기 → 설비·시각·내용 뽑기 → 조치가 필요한 건만 골라 메일 초안 쓰기**

처럼 매번 같은 순서로 처리합니다. 이런 일은 LLM에게 순서까지 맡기기보다, **사람이 순서를 정해 두고 각 단계에서만 LLM을 쓰는 것**이 더 예측 가능하고 비용도 적게 듭니다.

LangChain에서는 이렇게 단계를 이어 붙인 것을 **체인(Chain)** 또는 **파이프라인**이라고 부르고, `|` 기호로 연결합니다. (LCEL, LangChain Expression Language)

이번 노트북에서 다룰 내용

1. 프롬프트 템플릿
2. `|` 로 단계 연결하기
3. 구조화된 출력 (정해진 형식으로 답 받기)
4. 여러 체인 이어 붙이기
5. 병렬 실행 / 파이썬 함수 끼워 넣기
6. 조건에 따라 다른 체인 실행하기 (라우팅)
7. 여러 건 한 번에 처리하기, 스트리밍
8. 실패 대비 (재시도, 대체 모델)
9. 체인을 에이전트의 도구로 쓰기

In [ ]:
from langchain_openai import ChatOpenAI

import os
from dotenv import load_dotenv, find_dotenv

# .env 파일로부터 api_key를 불러오는 코드
load_dotenv(find_dotenv())

base_url = os.getenv("gpt_4o_mini_BASE_URL")

model = ChatOpenAI(
    model="gpt-4o-mini",
    base_url=base_url,
    temperature=0,  # 파이프라인은 같은 입력에 같은 결과가 나오는 것이 좋으므로 0으로 둡니다.
)

### 1. 프롬프트 템플릿 : `ChatPromptTemplate`

지금까지는 프롬프트를 매번 문자열로 직접 만들었습니다. 같은 형식의 요청을 반복한다면 **빈칸이 있는 양식**을 만들어 두고 빈칸만 채우는 편이 편합니다.

`{중괄호}` 안에 적은 이름이 빈칸이 됩니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 반도체 Fab ETCH 파트의 엔지니어를 돕는 어시스턴트다. 답변은 {style}로 작성한다."),
    ("user", "다음 교대 인수인계 메모를 요약해줘.\n\n{memo}"),
])

# 빈칸을 채우면 LLM에 보낼 메시지 목록이 만들어집니다. (아직 LLM을 호출하지 않습니다.)
filled = prompt.invoke({"style": "3줄 요약", "memo": "03호기 CH_B 압력 경고 발생"})

for message in filled.messages:
    print(f"[{message.type}] {message.content}")

### 2. `|` 로 단계 연결하기

`프롬프트 | 모델 | 출력 파서` 처럼 `|` 로 이어 붙이면, 왼쪽 단계의 결과가 오른쪽 단계의 입력으로 들어갑니다.

* `prompt` : 빈칸을 채워 메시지 목록을 만듦
* `model` : 메시지를 받아 AI 메시지(`AIMessage`)를 돌려줌
* `StrOutputParser()` : AI 메시지에서 답변 글자(`content`)만 꺼냄

이렇게 만든 `chain`도 `model`과 똑같이 `.invoke()`로 실행합니다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

summary_chain = prompt | model | StrOutputParser()

memo = """■ 08/16(일) 야간 / 박서진
01호기 : 특이X
02호기 : 특이X
03호기 :
- CH_B TV POSITION DEVIATION warn 22:31, 22:57 두번 뜸
- lot 끊기는 타이밍에 TV auto learn 돌림 (23:48~00:02). learn 후 warn 해제
- lot 대기 중이라 learn 전 값 백업은 못함
- learn 이후 P가 좀 높게 나오는 듯 (B07 기준 47~48, 원래 43 근처). UCL warn 뜨는데 I/L은 안 걸려서 진행 중
인계 : 03B P 추이 주간에 한번 봐주세요. 저는 내일부터 휴가(~8/24)"""

result = summary_chain.invoke({"style": "3줄 요약", "memo": memo})

print(result)  # AIMessage가 아니라 답변 글자만 바로 받을 수 있습니다.

**실습** : `style` 값을 바꿔서 결과가 어떻게 달라지는지 확인해 보세요. (예: `"파트장 보고용 한 문장"`, `"표 형식"`)

In [ ]:
result = summary_chain.invoke({"style": "", "memo": memo})
print(result)

### 3. 구조화된 출력 : `with_structured_output()`

요약문은 사람이 읽기엔 좋지만, 다음 단계의 코드가 사용하기엔 불편합니다. "설비가 무엇인지", "조치가 필요한지"를 코드에서 꺼내 쓰려면 **정해진 형식**으로 답을 받아야 합니다.

`pydantic`으로 원하는 형식(칸 이름, 자료형, 설명)을 정의하고 `model.with_structured_output()`에 넘겨주면, LLM이 그 형식에 맞춰 답합니다.

* `Field(description=...)` 의 설명은 도구의 docstring처럼 LLM이 읽는 안내문입니다.
* 값이 없을 수도 있는 칸은 `str | None` 처럼 적습니다.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


class ShiftEvent(BaseModel):
    """인수인계 메모에 적힌 사건 1건"""
    equipment_id: str = Field(description="설비 ID. '03호기', '03B' 같은 줄임말은 'ETCH_03'처럼 바꿔 적는다.")
    chamber_id: str | None = Field(description="챔버 ID. '03B'는 CH_B. 챔버가 없으면 null")
    time: str | None = Field(description="발생 시각 HH:MM. 없으면 null")
    event: str = Field(description="무슨 일이 있었는지 한 문장 요약")
    need_follow_up: bool = Field(description="다음 교대나 다른 담당자가 확인·조치해야 하면 true")


class ShiftNote(BaseModel):
    """교대 인수인계 메모 1건"""
    date: str = Field(description="날짜 MM/DD")
    shift: Literal["주간", "오후", "야간"]
    writer: str = Field(description="작성자 이름")
    events: list[ShiftEvent] = Field(description="특이사항이 있는 사건 목록. '특이X', '특이사항 없음'은 넣지 않는다.")


extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 ETCH 파트 교대 인수인계 메모를 정리하는 도우미다. 메모에 적힌 내용만 사용한다."),
    ("user", "{memo}"),
])

extract_chain = extract_prompt | model.with_structured_output(ShiftNote)

note = extract_chain.invoke({"memo": memo})

print(type(note))  # 문자열이 아니라 ShiftNote 객체로 받습니다.
print(note.date, note.shift, note.writer)
for event in note.events:
    print(event)

결과가 파이썬 객체이기 때문에 코드에서 바로 걸러낼 수 있습니다. 조치가 필요한 사건만 골라 봅시다.

In [ ]:
follow_ups = [e for e in note.events if e.need_follow_up]

for e in follow_ups:
    print(f"{e.equipment_id} {e.chamber_id or ''} {e.time or ''} : {e.event}")

**실습** : `ShiftEvent`에 `category` 칸을 추가해서 사건을 `"알람"`, `"작업"`, `"공정 변화"`, `"기타"` 중 하나로 분류하게 만들어 보세요. (`Literal`을 사용합니다.)

In [ ]:
# ShiftEvent, ShiftNote를 다시 정의한 뒤 extract_chain을 다시 만들고 실행해 봅니다.

### 4. 여러 체인 이어 붙이기

앞에서 만든 **추출 체인**의 결과를 받아 **메일 초안 체인**에 넘겨 봅시다.

체인끼리 이을 때는 앞 체인의 결과를 다음 프롬프트의 빈칸 이름에 맞춰 줘야 합니다. `RunnablePassthrough.assign()`은 **입력은 그대로 두고 새 값을 하나 추가**합니다.

```
{"memo": ...}
   → assign(note=추출 체인)   → {"memo": ..., "note": ShiftNote}
   → 메일 프롬프트 | 모델 | 파서 → "메일 초안"
```

In [ ]:
from langchain_core.runnables import RunnablePassthrough

mail_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 ETCH 파트 엔지니어다. 다음 교대 담당자에게 보낼 확인 요청 메일을 짧게 쓴다. "
               "조치가 필요한 사건만 다루고, 설비·챔버·시각을 빠뜨리지 않는다."),
    ("user", "정리된 인수인계 내용:\n{note}\n\n원문 메모:\n{memo}"),
])

mail_chain = (
    RunnablePassthrough.assign(note=extract_chain)  # 1단계: 메모에서 사건 추출
    | mail_prompt                                    # 2단계: 추출 결과 + 원문으로 프롬프트 채우기
    | model
    | StrOutputParser()
)

print(mail_chain.invoke({"memo": memo}))

### 5-1. 동시에 여러 작업 하기 : `RunnableParallel`

같은 메모로 **요약**, **위험도 판단**, **추출**을 각각 해야 한다면 하나씩 차례로 실행할 필요가 없습니다. `RunnableParallel`로 묶으면 동시에 실행되고, 결과는 이름표가 붙은 딕셔너리로 돌아옵니다.

In [ ]:
from langchain_core.runnables import RunnableParallel

risk_prompt = ChatPromptTemplate.from_messages([
    ("system", "메모를 읽고 생산에 영향을 줄 위험도를 '높음/보통/낮음' 중 하나와 이유 한 문장으로 답한다."),
    ("user", "{memo}"),
])

analysis = RunnableParallel(
    summary=prompt | model | StrOutputParser(),
    risk=risk_prompt | model | StrOutputParser(),
    note=extract_chain,
)

result = analysis.invoke({"style": "한 문장", "memo": memo})

print("요약   :", result["summary"])
print("위험도 :", result["risk"])
print("사건 수:", len(result["note"].events))

### 5-2. 파이썬 함수 끼워 넣기 : `RunnableLambda`

파이프라인의 모든 단계가 LLM일 필요는 없습니다. 파일 읽기, DB 조회, 값 계산처럼 **코드로 정확하게 할 수 있는 일은 코드로** 하는 것이 좋습니다.

일반 함수를 `RunnableLambda`로 감싸면 `|` 로 연결할 수 있습니다. 아래는 날짜와 교대를 받아 메모 원문을 찾아오는 함수를 파이프라인 앞에 붙인 예시입니다.

In [ ]:
from langchain_core.runnables import RunnableLambda

# 실제 업무에서는 파일이나 DB에서 읽어오는 부분입니다. 여기서는 간단히 딕셔너리로 대신합니다.
MEMO_DB = {
    ("08/16", "야간"): memo,
    ("08/17", "주간"): """■ 08/17(월) 주간 / 이한결
1. 03B P 확인 : 레시피 4개 전부 평소보다 +5 정도 높음. lot마다 UCL warn 뜨는 중
2. 새벽 lot 계측 나왔는데 03B D09_02 불량 7매. 다른 lot도 평소보다 많음
3. 12:05부터 03B 점검 들어감 (leak rate, CM gauge zero, 파티클). 오후조 인계
4. 02호기 RCP_B07 Rev.15 09:00 적용 완료 (김도윤 프로). 첫 lot CD in-spec
5. 파트장님께 03B 건 구두 보고함""",
}


def load_memo(inputs: dict) -> dict:
    """날짜와 교대를 받아 메모 원문을 찾아서 돌려줌"""
    text = MEMO_DB.get((inputs["date"], inputs["shift"]), "해당 교대의 메모가 없습니다.")
    return {"memo": text}


memo_to_mail = RunnableLambda(load_memo) | mail_chain

print(memo_to_mail.invoke({"date": "08/17", "shift": "주간"}))

### 6. 조건에 따라 다른 체인 실행하기 (라우팅)

질문 종류에 따라 처리 방법이 달라야 할 때가 있습니다. 예를 들어

* **데이터 질문** ("03B 불량률 얼마야?") → SQL 조회가 필요
* **문서 질문** ("03B 압력 경고 언제부터 떴어?") → 문서 검색이 필요
* **일반 질문** ("TV가 뭐야?") → LLM이 바로 답해도 됨

먼저 **분류 체인**으로 질문 종류를 정하고, 그 결과에 따라 다른 체인을 실행합니다. 함수가 체인을 돌려주면 LangChain이 그 체인을 이어서 실행합니다.

이 구조는 뒤의 멀티에이전트 노트북에서 **라우터(Router) 패턴**으로 다시 등장합니다.

In [ ]:
class Route(BaseModel):
    category: Literal["data", "document", "general"] = Field(
        description="data: 불량률·압력 등 공정 수치 조회가 필요한 질문 / "
                    "document: 설비 이력·인수인계 메모 확인이 필요한 질문 / "
                    "general: 용어 설명 등 일반 질문"
    )


classify_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "ETCH 파트 엔지니어의 질문을 분류한다."),
        ("user", "{question}"),
    ])
    | model.with_structured_output(Route)
)

# 종류별 처리 체인 (여기서는 흐름만 보기 위해 간단한 프롬프트로 대신합니다.)
data_chain = ChatPromptTemplate.from_template(
    "[데이터 담당] 다음 질문에 답하려면 어떤 SQL 조회가 필요한지 설명해줘: {question}") | model | StrOutputParser()
doc_chain = ChatPromptTemplate.from_template(
    "[문서 담당] 다음 질문에 답하려면 어떤 문서를 어떤 검색어로 찾아야 하는지 설명해줘: {question}") | model | StrOutputParser()
general_chain = ChatPromptTemplate.from_template(
    "반도체 ETCH 공정 엔지니어에게 설명하듯 답해줘: {question}") | model | StrOutputParser()


def route(inputs: dict):
    """분류 결과에 맞는 체인을 골라서 돌려줌"""
    category = inputs["route"].category
    print(f"(분류 결과: {category})")
    if category == "data":
        return data_chain
    if category == "document":
        return doc_chain
    return general_chain


router_chain = RunnablePassthrough.assign(route=classify_chain) | RunnableLambda(route)

for q in ["8월 ETCH_03 CH_B 불량률 알려줘", "03B 압력 경고는 언제부터 떴어?", "TV auto learn이 뭐야?"]:
    print("질문 :", q)
    print(router_chain.invoke({"question": q}), "\n")

### 7. 여러 건 한 번에 처리하기 / 스트리밍

**`.batch()`** : 입력 목록을 넘기면 동시에 처리해서 결과 목록을 돌려줍니다. 한 달치 메모를 한 번에 정리할 때처럼 같은 작업을 여러 번 할 때 사용합니다.

`max_concurrency`로 동시에 보낼 요청 수를 제한할 수 있습니다. (API 사용량 제한이 있을 때 유용합니다.)

In [ ]:
memos = list(MEMO_DB.values())

notes = extract_chain.batch([{"memo": m} for m in memos], config={"max_concurrency": 2})

for n in notes:
    print(n.date, n.shift, "-", [e.event for e in n.events if e.need_follow_up])

**`.stream()`** : 답변이 다 만들어질 때까지 기다리지 않고, 만들어지는 대로 조금씩 받아옵니다. 챗봇 화면에서 글자가 타이핑되듯 나오는 것이 이 방식입니다.

In [ ]:
for chunk in summary_chain.stream({"style": "파트장 보고용 5줄 요약", "memo": memo}):
    print(chunk, end="", flush=True)

### 8. 실패 대비 : 재시도와 대체 모델

업무에서 사용하는 파이프라인은 API가 잠깐 응답하지 않거나, 모델 이름이 바뀌는 등의 이유로 실패할 수 있습니다.

* **`.with_retry()`** : 실패하면 정해진 횟수만큼 다시 시도
* **`.with_fallbacks([...])`** : 실패하면 준비해 둔 다른 모델(또는 체인)로 대신 실행

아래는 일부러 **존재하지 않는 모델 이름**을 넣어 실패하게 만든 뒤, 대체 모델로 넘어가는지 확인하는 예시입니다.

In [ ]:
broken_model = ChatOpenAI(model="없는-모델", base_url=base_url, max_retries=0)

safe_model = broken_model.with_fallbacks([model])  # 실패하면 gpt-4o-mini로 대신 실행

safe_chain = prompt | safe_model | StrOutputParser()

print(safe_chain.invoke({"style": "한 문장", "memo": memo}))

In [ ]:
# 재시도: 최대 3번까지 시도합니다. (네트워크 오류처럼 다시 하면 성공할 수 있는 경우에 사용)
retry_chain = prompt | model.with_retry(stop_after_attempt=3) | StrOutputParser()

print(retry_chain.invoke({"style": "한 문장", "memo": memo}))

### 9. 체인을 에이전트의 도구로 쓰기

체인과 에이전트는 함께 쓸 수 있습니다. **순서가 정해진 부분은 체인으로 만들고, 그 체인을 도구로 감싸서 에이전트에게 주면** 에이전트는 "언제 쓸지"만 판단하면 됩니다.

아래 에이전트는 사용자의 요청을 보고, 필요할 때 `draft_follow_up_mail` 도구(= 앞에서 만든 `memo_to_mail` 체인)를 호출합니다.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def draft_follow_up_mail(date: str, shift: Literal["주간", "오후", "야간"]) -> str:
    """지정한 날짜(MM/DD)와 교대의 인수인계 메모를 읽고, 조치가 필요한 사건에 대한 확인 요청 메일 초안을 만든다."""
    return memo_to_mail.invoke({"date": date, "shift": shift})


agent = create_agent(
    model,
    tools=[draft_follow_up_mail],
    system_prompt="너는 ETCH 파트 업무를 돕는 어시스턴트다.",
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "8월 16일 야간 인수인계 보고 다음 조에 보낼 메일 좀 써줘"}]}
)

for message in response["messages"]:
    tool_calls = getattr(message, "tool_calls", None)
    if tool_calls:
        print("도구 호출 :", [(t["name"], t["args"]) for t in tool_calls])
print("\n답변 :\n", response["messages"][-1].content)

### 정리 : 체인과 에이전트, 언제 무엇을 쓸까?

| | 체인 (파이프라인) | 에이전트 |
|---|---|---|
| 순서 | 사람이 미리 정함 | LLM이 상황을 보고 정함 |
| 결과 | 매번 비슷하게 나옴 | 질문에 따라 경로가 달라짐 |
| 비용·시간 | 호출 횟수가 정해져 있음 | 도구를 몇 번 부를지 알 수 없음 |
| 어울리는 일 | 정형화된 반복 업무 (메모 정리, 보고서 양식 채우기, 분류) | 질문이 다양하고 조사 경로가 매번 다른 업무 (원인 조사, 문의 대응) |

실무에서는 둘을 섞는 경우가 많습니다. **정해진 부분은 체인으로 고정하고, 판단이 필요한 부분만 에이전트에게 맡기는 것**이 일반적입니다.

## 실습

아래 순서로 동작하는 **알람 보고 파이프라인**을 만들어 보세요.

1. 입력 : 인수인계 메모 원문 (`memo`)
2. `ShiftNote` 형식으로 사건 추출 (`extract_chain` 재사용)
3. 파이썬 함수(`RunnableLambda`)로 `need_follow_up`이 `True`인 사건만 남기기
4. 남은 사건이 없으면 `"보고할 사건 없음"`을 돌려주고, 있으면 파트장 보고용 3줄 요약을 작성

힌트 : 4번은 6장의 라우팅처럼 함수 안에서 조건에 따라 다른 체인(또는 문자열)을 돌려주면 됩니다.

In [ ]:
def keep_follow_ups(note):
    # 조치가 필요한 사건만 남기는 코드를 작성합니다.
    pass


def report_or_skip(events):
    # 사건이 없으면 "보고할 사건 없음", 있으면 요약 체인을 돌려줍니다.
    pass


alarm_report_chain = None  # extract_chain | RunnableLambda(...) | RunnableLambda(...)

# print(alarm_report_chain.invoke({"memo": memo}))
# print(alarm_report_chain.invoke({"memo": "■ 08/02(일) 야간 / 이한결\n특이사항 없음"}))